# 09 — SAHAR Evaluation — Training-Free Attention Refinement (ViT-Base/16, CRC Histology)

Tests a **training-free adaptation of SAHAR** (Scale-Aware Hierarchical Attention
Refinement) against raw attention and standard Attention Rollout on a trained
**ViT-Base/16**, CRC-VAL-HE-7K test set (9 classes), using **Insertion / Deletion AUC**.

## Why training-free, and why insertion/deletion only

The original SAHAR is a **learnable** module (scale projections, cross-attention
linear layers, a 7×7 gating conv, fusion weights) trained with a binary-cross-entropy
**faithfulness loss against radiologist lesion masks**, on a Swin-V2 backbone, on
chest X-ray data. Our setting differs on three points that make the method
non-transferable as-is:

1. **No masks.** CRC-HE patches are single-class tissue crops — there are no lesion
   masks, so SAHAR's supervised faithfulness loss cannot be instantiated.
2. **Author metrics need masks.** Pointing Game, IoU and EBPG all require a ground-truth
   region inside the image; CRC has none. **Insertion/Deletion is the only applicable
   faithfulness metric here**, which is what we evaluate.
3. **Backbone.** SAHAR targets hierarchical Swin-V2; we use flat ViT-B/16 (notebook 08's
   model). The four mechanisms adapt cleanly to ViT's 14×14 patch grid + CLS readout.

We therefore keep only SAHAR's **parameter-free inductive biases** (learnable parts
replaced by deterministic operators) and ask whether that mechanism, on its own,
improves insertion/deletion faithfulness over the raw attention it refines:

| Mechanism | Learnable original | Training-free surrogate |
|---|---|---|
| MRTP (multi-resolution pooling) | + linear projection | average pooling, identity projection |
| SACA (scale-aware cross-attn) | — (cosine-similarity prior) | **kept as-is** (already parameter-free) |
| RGAG (resolution-guided gating) | learned 7×7 conv | fixed low-pass gate |
| LWA (weighted fusion) | learned weights (init 0) | uniform mean (== the init) |

**Ablation grid** (mirrors the paper's Table 4): `raw_attention`, `rollout`,
`sahar_full`, `sahar_no_saca`, `sahar_no_rgag`, `sahar_single_scale`.

**Edit only Cell 2** (`USER CONFIG`). **Environment**: Kaggle GPU T4 / P100 — Phase 1 only.

In [ ]:
# 0. Kaggle setup
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical
!pip install -q timm albumentations loguru
!pip install -q PyDrive2
!pip install -q scikit-learn

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
# ============================================================
# USER CONFIG — edit this cell only
# ============================================================

MODEL_NAME = "vit_base"   # this notebook targets ViT-Base/16 CLS-token readout

# Google Drive folder containing the vit_base checkpoint (and where results are uploaded)
DRIVE_FOLDER_ID = "190QMsyYCtNo5xXOEm-CWWQCDm4jfPGsu"  # vit_base <-- update as needed

# Dataset subset
N_IMAGES_PER_CLASS = 30
BATCH_SIZE          = 8
NUM_WORKERS         = 2
IMAGE_SIZE          = 224
SEED                = 42

# SAHAR (training-free) hyperparameters
SAHAR_LAYER   = 11          # transformer block to read attention/tokens from (0..11)
SAHAR_SCALES  = (2, 4, 0)   # MRTP pooling kernels: micro=2, meso=4, macro=0 (global)
GATE_KERNEL   = 7           # RGAG low-pass kernel (odd); paper's conv is 7x7
GATE_GAIN     = 10.0        # RGAG sigmoid slope (fixed, replaces the learned conv)

# Variants (ablation grid, paper Table 4 adapted to the training-free setting)
VARIANTS = [
    "raw_attention",       # base CLS attention, no refinement (SAHAR baseline)
    "rollout",             # standard attention rollout (external baseline)
    "sahar_full",          # MRTP + SACA + RGAG + LWA
    "sahar_no_saca",       # ablate the anatomical-prior recalibration
    "sahar_no_rgag",       # ablate the gating
    "sahar_single_scale",  # ablate multi-scale pooling
]

# Faithfulness (patch-level insertion/deletion, Gaussian-blur baseline)
FAITH_N_STEPS = 49
N_VIZ         = 6

# Paths (Kaggle)
TRAINVAL_ROOT_STR = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K"
TEST_ROOT_STR     = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K"
PROJECT_ROOT      = "/kaggle/working/xai-vit-medical"
CKPT_LOCAL        = "/kaggle/input/models/youssefnouiouar1/<UPDATE_ME>/pytorch/default/1/vit_base_patch16_best.pth"  # <-- update to the vit_base Kaggle model path

In [ ]:
import csv
import gc
import json
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.notebook import tqdm

assert MODEL_NAME == "vit_base", "This notebook targets ViT-B/16 (CLS readout)."

for p in (PROJECT_ROOT, f"{PROJECT_ROOT}/notebooks"):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.data.crc_dataset import CRCHistologyDataset, CRCSplits, DEFAULT_CRC_CLASSES
from src.utils.seed import set_seed
from sahar import SAHARCapture, SAHARConfig, sahar_saliency, variant_config
from slr_ar import AttentionCatcher, SLRARConfig, build_rollout, cls_attribution_map

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = list(DEFAULT_CRC_CLASSES)
NUM_CLASSES = len(CLASS_NAMES)

GRID  = IMAGE_SIZE // 16   # 14 for 224px ViT-B/16
PATCH = 16
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

SAVE_DIR = Path(f"{PROJECT_ROOT}/outputs/sahar/{MODEL_NAME}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
TRAINVAL_ROOT = Path(TRAINVAL_ROOT_STR)
TEST_ROOT     = Path(TEST_ROOT_STR)

print(f"Device  : {DEVICE}")
print(f"Classes : {CLASS_NAMES}")
print(f"Grid    : {GRID}x{GRID} patches")
print(f"Output  : {SAVE_DIR}")
print(f"Checkpoint path: {CKPT_LOCAL}")

In [ ]:
# ── Build model ────────────────────────────────────────────────────────────

def load_state(model: nn.Module, ckpt_path: str) -> dict:
    state = torch.load(ckpt_path, map_location="cpu")
    if isinstance(state, dict) and "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"])
        return {k: v for k, v in state.items() if k != "model_state_dict"}
    if isinstance(state, dict) and "state_dict" in state:
        cleaned = {k.replace("module.", "", 1): v for k, v in state["state_dict"].items()}
        model.load_state_dict(cleaned, strict=False)
        return {}
    cleaned = {k.replace("module.", "", 1): v for k, v in state.items()}
    model.load_state_dict(cleaned, strict=False)
    return {}


model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=NUM_CLASSES,
    global_pool="token",   # CLS-token readout
    drop_path_rate=0.0,
)
meta = load_state(model, CKPT_LOCAL)
model = model.to(DEVICE).eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model            : {MODEL_NAME}")
print(f"Parameters       : {total_params:,}")
print(f"Checkpoint epoch : {meta.get('epoch', '?')}")
print(f"Checkpoint acc   : {meta.get('val_acc', '?')}")

In [ ]:
# ── Dataset & test subset ──────────────────────────────────────────────────
crc_splits = CRCSplits(
    trainval_root=TRAINVAL_ROOT, test_root=TEST_ROOT,
    classes=tuple(CLASS_NAMES), val_ratio=0.25, random_state=SEED,
)
test_dataset = CRCHistologyDataset(split="test", splits=crc_splits,
                                   image_size=IMAGE_SIZE, return_id=True)

class_counts = defaultdict(int)
subset_indices = []
for idx, label in enumerate(test_dataset.labels):
    lbl = int(label)
    if class_counts[lbl] < N_IMAGES_PER_CLASS:
        subset_indices.append(idx); class_counts[lbl] += 1
    if all(class_counts[c] >= N_IMAGES_PER_CLASS for c in range(NUM_CLASSES)):
        break

eval_loader = DataLoader(Subset(test_dataset, subset_indices), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=torch.cuda.is_available())
print("Images per class:")
for ci, cn in enumerate(CLASS_NAMES):
    print(f"  {cn}: {class_counts[ci]}")
print(f"Total: {len(subset_indices)} | Batches: {len(eval_loader)}")

In [ ]:
# ── SAHAR base config ──────────────────────────────────────────────────────
sahar_base = SAHARConfig(
    layer=SAHAR_LAYER, scales=SAHAR_SCALES,
    gate_kernel=GATE_KERNEL, gate_gain=GATE_GAIN,
    num_extra_tokens=1,
)
roll_cfg = SLRARConfig()   # defaults; the "rollout" variant ignores SNB/GTS
print(sahar_base)
print(f"Reading attention/tokens from block {SAHAR_LAYER}")

In [ ]:
# ── Faithfulness metric (patch-level insertion/deletion, blur baseline) ────

def _blur(x: torch.Tensor, sigma_px: int = 11) -> torch.Tensor:
    k = sigma_px if sigma_px % 2 == 1 else sigma_px + 1
    coords = torch.arange(k, dtype=x.dtype, device=x.device) - k // 2
    g = torch.exp(-(coords ** 2) / (2 * (k / 3.0) ** 2))
    g = (g / g.sum()).view(1, 1, 1, k)
    C = x.shape[1]
    x = F.conv2d(F.pad(x, (k // 2,) * 2 + (0, 0), mode="reflect"), g.expand(C, 1, 1, k), groups=C)
    x = F.conv2d(F.pad(x, (0, 0) + (k // 2,) * 2, mode="reflect"), g.transpose(-1, -2).expand(C, 1, k, 1), groups=C)
    return x


@torch.no_grad()
def insertion_deletion_auc(model, x, sal, target, mode, n_steps=FAITH_N_STEPS, patch=PATCH, grid=GRID, fwd_bs=32):
    """AUC over patch-level insertion/deletion for one image. sal: (1,grid,grid)."""
    device = x.device
    order = torch.argsort(sal.reshape(-1), descending=True)
    n_patch = grid * grid
    per_step = max(1, n_patch // n_steps)
    steps = list(range(0, n_patch + 1, per_step))
    if steps[-1] != n_patch:
        steps.append(n_patch)
    base = _blur(x) if mode == "insertion" else torch.zeros_like(x)
    start, fill = (base, x) if mode == "insertion" else (x, base)
    masks = []
    for s in steps:
        m = torch.zeros(n_patch, device=device); m[order[:s]] = 1.0
        masks.append(F.interpolate(m.reshape(1, 1, grid, grid), scale_factor=patch, mode="nearest"))
    masks = torch.cat(masks, dim=0)
    imgs = start * (1 - masks) + fill * masks
    probs = []
    for i in range(0, imgs.shape[0], fwd_bs):
        probs.append(model(imgs[i:i + fwd_bs]).softmax(-1)[:, target])
    probs = torch.cat(probs).cpu().numpy()
    return float(np.trapz(probs, np.array(steps) / n_patch))


def overlay_pair(image_chw, saliency_hw):
    img = image_chw.detach().cpu().numpy().transpose(1, 2, 0)
    mean = IMAGENET_MEAN.view(3).numpy(); std = IMAGENET_STD.view(3).numpy()
    img = np.clip(img * std + mean, 0.0, 1.0)
    sal = saliency_hw.astype(np.float32)
    sal = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)
    sal_up = F.interpolate(torch.from_numpy(sal)[None, None], size=img.shape[:2],
                           mode="bilinear", align_corners=False)[0, 0].numpy()
    return img, sal_up


def compute_saliency(variant, attn_L, tokens_L, attns_all):
    """Return (B, grid, grid) saliency for a variant."""
    if variant == "rollout":
        R, _ = build_rollout(attns_all, "rollout", roll_cfg)
        return cls_attribution_map(R, grid=GRID)
    return sahar_saliency(attn_L, tokens_L, variant_config(variant, sahar_base))


print("Evaluation helpers ready")

In [ ]:
# ── Main evaluation loop — insertion/deletion per variant ──────────────────
catcher_roll = AttentionCatcher(model)          # all layers (for rollout)
cap = SAHARCapture(model, sahar_base.layer)     # layer L attn + tokens (for SAHAR)

acc = {v: {"ins": [], "del": []} for v in VARIANTS}
per_image_rows = []
viz_x, viz_y = None, None
viz_sal = {v: None for v in VARIANTS}
n_correct = 0
n_total = 0

for bi, (images, labels, image_ids) in enumerate(tqdm(eval_loader, desc="SAHAR eval")):
    images = images.to(DEVICE, non_blocking=True)
    catcher_roll.clear()
    catcher_roll.resume()
    logits = model(images)          # clean forward: fills both catchers
    catcher_roll.pause()            # masked forwards below must NOT be captured (rollout list would leak)
    preds = logits.argmax(-1)
    n_correct += int((preds.cpu() == labels).sum())
    n_total += images.shape[0]

    attns_all = list(catcher_roll.attns)   # L tensors (B,H,N,N)
    attn_L, tokens_L = cap.attn, cap.tokens

    sal = {v: compute_saliency(v, attn_L, tokens_L, attns_all) for v in VARIANTS}

    for v in VARIANTS:
        for i in range(images.shape[0]):
            t = int(preds[i])
            ins = insertion_deletion_auc(model, images[i:i+1], sal[v][i:i+1], t, "insertion")
            dele = insertion_deletion_auc(model, images[i:i+1], sal[v][i:i+1], t, "deletion")
            acc[v]["ins"].append(ins); acc[v]["del"].append(dele)
            per_image_rows.append({
                "image_id": image_ids[i], "label_idx": int(labels[i]),
                "label_name": CLASS_NAMES[int(labels[i])], "pred_idx": t,
                "pred_name": CLASS_NAMES[t], "variant": v,
                "insertion_auc": ins, "deletion_auc": dele, "faithfulness": ins - dele,
            })

    if bi == 0:
        viz_x = images[:N_VIZ].detach().cpu()
        viz_y = [CLASS_NAMES[int(c)] for c in labels[:N_VIZ]]
        for v in VARIANTS:
            viz_sal[v] = sal[v][:N_VIZ].detach().cpu()

cap.remove()
catcher_roll.remove()
model_accuracy = n_correct / n_total
print(f"Model accuracy on subset: {model_accuracy:.4f} ({n_correct}/{n_total})")

In [ ]:
# ── Summary table & save ───────────────────────────────────────────────────
summary = {}
print("=" * 66)
print(f"{'Variant':<22}{'Insertion':>12}{'Deletion':>12}{'Faithfulness':>16}")
print("-" * 66)
for v in VARIANTS:
    ins = float(np.mean(acc[v]["ins"]))
    dele = float(np.mean(acc[v]["del"]))
    summary[v] = {"insertion_auc": ins, "deletion_auc": dele, "faithfulness": ins - dele}
    print(f"{v:<22}{ins:>12.4f}{dele:>12.4f}{ins - dele:>16.4f}")
print("=" * 66)
print("Higher insertion, lower deletion, higher faithfulness = better explanation.")

best = max(VARIANTS, key=lambda v: summary[v]["faithfulness"])
raw = summary["raw_attention"]["faithfulness"]
print(f"\nBest variant: {best} (faithfulness {summary[best]['faithfulness']:.4f})")
print(f"SAHAR-full vs raw attention: {summary['sahar_full']['faithfulness'] - raw:+.4f} faithfulness")

with open(SAVE_DIR / "summary.json", "w") as f:
    json.dump({
        "model": MODEL_NAME, "model_accuracy": model_accuracy, "n_images": n_total,
        "config": {"layer": SAHAR_LAYER, "scales": list(SAHAR_SCALES),
                   "gate_kernel": GATE_KERNEL, "gate_gain": GATE_GAIN},
        "results": summary,
    }, f, indent=2)

per_image_csv = SAVE_DIR / "per_image_results.csv"
with open(per_image_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(per_image_rows[0].keys()))
    w.writeheader(); w.writerows(per_image_rows)
print(f"Saved: {SAVE_DIR / 'summary.json'}")
print(f"Saved: {per_image_csv}")

In [ ]:
# ── Qualitative overlays: input + key variants ────────────────────────────
show = [v for v in ["raw_attention", "rollout", "sahar_full"] if v in VARIANTS]
n = viz_x.shape[0]
fig, axes = plt.subplots(n, len(show) + 1, figsize=(3.2 * (len(show) + 1), 3.2 * n))
axes = np.atleast_2d(axes)
mean = IMAGENET_MEAN.view(3).numpy(); std = IMAGENET_STD.view(3).numpy()
for r in range(n):
    img = np.clip(viz_x[r].permute(1, 2, 0).numpy() * std + mean, 0, 1)
    axes[r, 0].imshow(img)
    axes[r, 0].set_ylabel(viz_y[r], fontsize=9, rotation=0, labelpad=40, va="center")
    if r == 0:
        axes[r, 0].set_title("input")
    for c, v in enumerate(show):
        img_u, sal_up = overlay_pair(viz_x[r], viz_sal[v][r].numpy())
        axes[r, c + 1].imshow(img_u)
        axes[r, c + 1].imshow(sal_up, cmap="jet", alpha=0.5)
        if r == 0:
            axes[r, c + 1].set_title(v)
    for a in axes[r]:
        a.set_xticks([]); a.set_yticks([])
fig.tight_layout()
fig.savefig(SAVE_DIR / "qualitative.png", dpi=150)
plt.show()

# Faithfulness bar chart
fig, ax = plt.subplots(figsize=(8, 4))
faith = [summary[v]["faithfulness"] for v in VARIANTS]
colors = ["tab:gray" if v in ("raw_attention", "rollout") else "tab:blue" for v in VARIANTS]
ax.bar(range(len(VARIANTS)), faith, color=colors)
ax.set_xticks(range(len(VARIANTS))); ax.set_xticklabels(VARIANTS, rotation=30, ha="right")
ax.set_ylabel("Faithfulness (Insertion − Deletion)")
ax.axhline(summary["raw_attention"]["faithfulness"], color="k", ls="--", lw=0.8, label="raw attention")
ax.set_title(f"{MODEL_NAME} — SAHAR ablation (higher = better)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(SAVE_DIR / "faithfulness_bars.png", dpi=150)
plt.show()
print(f"Saved: {SAVE_DIR / 'qualitative.png'}")
print(f"Saved: {SAVE_DIR / 'faithfulness_bars.png'}")

In [ ]:
# ── Upload results to Google Drive ─────────────────────────────────────────
RESULTS_DRIVE_FOLDER = DRIVE_FOLDER_ID

files_to_upload = [
    SAVE_DIR / "summary.json",
    SAVE_DIR / "per_image_results.csv",
    SAVE_DIR / "qualitative.png",
    SAVE_DIR / "faithfulness_bars.png",
]
for fpath in files_to_upload:
    fpath = Path(fpath)
    if not fpath.exists():
        print(f"  Not found (skipped): {fpath.name}"); continue
    drive_file = drive.CreateFile({"title": f"sahar_{MODEL_NAME}_{fpath.name}",
                                   "parents": [{"id": RESULTS_DRIVE_FOLDER}]})
    drive_file.SetContentFile(str(fpath)); drive_file.Upload()
    print(f"  Uploaded: sahar_{MODEL_NAME}_{fpath.name}")
print("Done.")

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup done.")